In [ ]:
# Cell 1: Thư viện & Khởi tạo Cấu hình (CFG)
import os
import shutil
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import glob
import warnings

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix
import torch.optim as optim
from collections import Counter

warnings.filterwarnings('ignore')

class CFG:
    # 1. Đường dẫn
    drive_folder = '/content/drive/MyDrive/eyepacs_6k'
    extract_dir  = '/content/dataset_full'
    csv_path     = '/content/full_labels.csv'
    save_path    = '/content/drive/MyDrive/efficientnet_b4_dr_best.pth'
    last_path    = '/content/drive/MyDrive/efficientnet_b4_dr_last.pth'

    img_size     = 380
    batch_size   = 128
    epochs       = 30
    lr           = 3e-4
    weight_decay = 1e-4
    mixup_alpha  = 0.1
    patience     = 5
    val_ratio    = 0.15
    max_class    = 4     # Các class: 0, 1, 2, 3, 4
    seed         = 42
    num_workers  = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(CFG.seed)
np.random.seed(CFG.seed)

In [ ]:
# Cell 2: Khôi phục Dữ liệu (Auto Extract)
def setup_dataset():
    if os.path.exists(CFG.extract_dir) and len(os.listdir(CFG.extract_dir)) >= 6:
        print(f"Dataset is ready at {CFG.extract_dir}. Skipping extraction.")
        return

    print(f"🔍 Scanning Drive folder: {CFG.drive_folder}...")
    os.makedirs(CFG.extract_dir, exist_ok=True)

    csv_files = glob.glob(os.path.join(CFG.drive_folder, '*.csv'))
    if csv_files:
        shutil.copy(csv_files[0], CFG.csv_path)
        print(f"Copied label file: {os.path.basename(csv_files[0])}")
    else:
        raise FileNotFoundError("Error: No .csv file found in the drive folder!")

    sub_zips = glob.glob(os.path.join(CFG.drive_folder, '*.zip'))
    print(f"Found {len(sub_zips)} zip archives. Starting sequential extraction...")

    local_temp_zip = '/content/temp_data.zip'
    for i, zip_path in enumerate(sub_zips, 1):
        print(f"Processing [{i}/{len(sub_zips)}]: {os.path.basename(zip_path)}", end='\r')
        shutil.copyfile(zip_path, local_temp_zip)
        !unzip -q -n {local_temp_zip} -d {CFG.extract_dir}
        os.remove(local_temp_zip)

    print("\nDataset setup complete! 35K images merged.")

setup_dataset()

Dataset is ready at /content/dataset_full. Skipping extraction.


In [ ]:
# Cell 3: Image Transformations & Ben Graham Filter
def crop_image_from_gray(img, tol=7):
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    rows = mask.any(1)
    cols = mask.any(0)
    if not rows.any() or not cols.any():
        return img   # guard: ảnh quá tối
    return img[np.ix_(rows, cols)]

def apply_ben_graham(path, sigma=10, size=CFG.img_size):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = crop_image_from_gray(img)                              # ← bước mới
    img = cv2.resize(img, (size, size))
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), sigma), -4, 128)
    mask = np.zeros(img.shape, dtype=np.uint8)
    cv2.circle(mask, (size // 2, size // 2), int(size * 0.47), (1, 1, 1), -1)
    return Image.fromarray((img * mask).astype(np.uint8))

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print("Transforms initialized.")

Transforms initialized.


In [ ]:
# Cell 4: Custom Dataset & DataLoader Generator (Cleaned)
class DRDataset(Dataset):
    def __init__(self, records, image_paths, transform=None):
        self.records, self.image_paths, self.transform = records, image_paths, transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        img_name, label = self.records[idx]
        img_id = str(img_name).strip().split('.')[0]
        if img_id not in self.image_paths:
            return None
        image = apply_ben_graham(self.image_paths[img_id])
        if image is None:
            return None
        if self.transform:
            image = self.transform(image)
        # Ép kiểu float32 chuẩn chỉnh cho Regression
        return image, torch.tensor(label, dtype=torch.float32)

def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    return torch.utils.data.dataloader.default_collate(batch) if batch else None

# Mapping ảnh
print("Mapping image paths...")
img_paths = {
    os.path.splitext(f)[0]: os.path.join(root, f)
    for root, _, files in os.walk(CFG.extract_dir)
    for f in files if f.lower().endswith(('.jpeg', '.jpg', '.png'))
}
print(f"  Found {len(img_paths):,} images.")

df          = pd.read_csv(CFG.csv_path)
all_records = [(row.iloc[0], row.iloc[1]) for _, row in df.iterrows()]
all_labels  = [r[1] for r in all_records]

train_rec, val_rec = train_test_split(
    all_records, test_size=CFG.val_ratio,
    stratify=all_labels, random_state=CFG.seed
)
print(f"Dataset Split -> Train: {len(train_rec):,} | Validation: {len(val_rec):,}")

# DataLoader thuần tự nhiên
train_loader = DataLoader(
    DRDataset(train_rec, img_paths, train_transforms),
    batch_size=CFG.batch_size,
    shuffle=True,                    # Đảo ngẫu nhiên, học theo tỷ lệ gốc
    num_workers=CFG.num_workers,
    collate_fn=safe_collate,
    pin_memory=True
)

val_loader = DataLoader(
    DRDataset(val_rec, img_paths, val_transforms),
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=safe_collate,
    pin_memory=True
)
print("DataLoaders ready.")

Mapping image paths...
  Found 35,126 images.
Dataset Split -> Train: 29,857 | Validation: 5,269
✅ DataLoaders ready.


In [ ]:
# Cell 5: EfficientNet-B4 Architecture (Regression Head)
def build_model():
    net = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)

    # Đóng băng xương sống
    for param in net.parameters():
        param.requires_grad = False

    # Mở băng 3 layer cuối để tinh chỉnh (Fine-tuning)
    for name, param in net.named_parameters():
        if any(k in name for k in ['features.6', 'features.7', 'features.8', 'classifier']):
            param.requires_grad = True

    # Thay đầu phân loại bằng đầu hồi quy (1 Output)
    in_features = net.classifier[1].in_features
    net.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, 1) # <--- 1 nơ-ron Regression
    )
    return net.to(device)

model = build_model()
print(f"Model built. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model built. Trainable params: 14,831,161


In [ ]:
# Cell 6: Checkpoint Loader & Surgery (Dual-Mode Resume)
start_epoch = 0
best_qwk = -1.0
is_regression_resume = False

if os.path.exists(CFG.save_path):
    print(f"Found existing checkpoint: {CFG.save_path}")
    checkpoint = torch.load(CFG.save_path, map_location=device, weights_only=False)

    # Kiểm tra cấu hình Dictionary nâng cao
    if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
        state_dict = checkpoint['model_state']

        # MẸO KAGGER: Kiểm tra số lượng nơ-ron đầu ra của file checkpoint cũ
        if 'classifier.4.weight' in state_dict and state_dict['classifier.4.weight'].shape[0] != 1:
            state_dict.pop('classifier.4.weight', None)
            state_dict.pop('classifier.4.bias', None)
            model.load_state_dict(state_dict, strict=False)
        else:

            model.load_state_dict(state_dict)
            start_epoch = checkpoint['epoch']
            best_qwk = checkpoint['best_qwk']
            is_regression_resume = True
            print(f"Resume Regression at Epoch {start_epoch}.")
    else:
        # Phòng trường hợp file .pth chỉ chứa state_dict thuần túy
        if 'classifier.4.weight' in checkpoint and checkpoint['classifier.4.weight'].shape[0] != 1:
            checkpoint.pop('classifier.4.weight', None)
            checkpoint.pop('classifier.4.bias', None)
            model.load_state_dict(checkpoint, strict=False)
        else:
            model.load_state_dict(checkpoint)
        print("Loaded raw state_dict.")
else:
    print("No existing checkpoint found.")

Found existing checkpoint: /content/drive/MyDrive/efficientnet_b4_dr_best.pth
Resume Regression at Epoch 10.


In [ ]:
# Cell 7: Loss Function & Optimizer
# Hàm mất mát tối ưu nhất cho Hồi quy (Regression)
import scipy.optimize
from functools import partial
criterion = nn.SmoothL1Loss()

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG.lr, weight_decay=CFG.weight_decay
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.epochs, eta_min=1e-6
)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

class OptimizedRounder:
    def __init__(self):
        self.coef_ = np.array([0.5, 1.5, 2.5, 3.5])

    def _predict(self, X, coef):
        X_p = np.full_like(X, 4, dtype=int)
        X_p[X < coef[3]] = 3
        X_p[X < coef[2]] = 2
        X_p[X < coef[1]] = 1
        X_p[X < coef[0]] = 0
        return X_p

    def _loss(self, coef, X, y):
        return -cohen_kappa_score(y, self._predict(X, coef), weights='quadratic')

    def fit(self, X, y):
        result     = scipy.optimize.minimize(
            partial(self._loss, X=X, y=y),
            self.coef_, method='nelder-mead'
        )
        self.coef_ = result.x
        best_qwk   = -result.fun
        print(f"  Optimal thresholds: {[f'{c:.3f}' for c in self.coef_]}")
        print(f"  Val QWK (optimized): {best_qwk:.4f}")
        return best_qwk

    def predict(self, X):
        return self._predict(X, self.coef_)

optR = OptimizedRounder()
print("Loss: SmoothL1 | Optimizer: AdamW | OptimizedRounder: ready.")

Loss: SmoothL1 | Optimizer: AdamW | OptimizedRounder: ready.


In [ ]:
# Cell 8: Training & Validation Engine
def mixup_data(x, y, alpha=CFG.mixup_alpha):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_one_epoch(model, loader, optimizer, criterion):
    """scheduler.step() được gọi ở vòng lặp chính — không gọi trong hàm này."""
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []

    for i, batch in enumerate(loader):
        if batch is None: continue
        images = batch[0].to(device)
        labels = batch[1].to(device)

        images, targets_a, targets_b, lam = mixup_data(images, labels)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images).view(-1)
            loss    = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds_np = np.clip(np.round(outputs.detach().cpu().numpy()), 0, CFG.max_class).astype(int)
        all_preds.extend(preds_np)
        all_labels.extend(labels.cpu().numpy().astype(int))

        if (i + 1) % 50 == 0:
            print(f"      - Batch [{i+1}/{len(loader)}] | Loss: {loss.item():.4f}")

    n = len(all_labels)
    return (running_loss / n,
            accuracy_score(all_labels, all_preds),
            cohen_kappa_score(all_labels, all_preds, weights='quadratic'))

@torch.no_grad()
def evaluate(model, loader, criterion):
    """Trả thêm raw_preds (float) để OptimizedRounder tối ưu ngưỡng."""
    model.eval()
    running_loss   = 0.0
    raw_preds, all_labels = [], []

    for batch in loader:
        if batch is None: continue
        images = batch[0].to(device)
        labels = batch[1].to(device)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images).view(-1)
            loss    = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        raw_preds.extend(outputs.detach().cpu().numpy())     # float thô
        all_labels.extend(labels.cpu().numpy().astype(int))

    # Round cứng để in nhanh
    preds_hard = np.clip(np.round(raw_preds), 0, CFG.max_class).astype(int)
    print("\n   [Confusion Matrix - Validation (hard round)]:")
    print(confusion_matrix(all_labels, preds_hard))

    n   = len(all_labels)
    qwk = cohen_kappa_score(all_labels, preds_hard, weights='quadratic')
    acc = accuracy_score(all_labels, preds_hard)
    return (running_loss / n, acc, qwk,
            np.array(raw_preds), np.array(all_labels))  # raw để rounder dùng


In [ ]:
# Cell 9: The Main Training Loop
patience_cnt = 0
history      = []

print(f"\nINIT TRAINING PIPELINE (REGRESSION) - STARTING EPOCH {start_epoch+1}...\n")

for epoch in range(start_epoch, CFG.epochs):
    print(f"Epoch [{epoch+1}/{CFG.epochs}]")

    tr_loss, tr_acc, tr_qwk = train_one_epoch(
        model, train_loader, optimizer, criterion
    )
    vl_loss, vl_acc, vl_qwk, val_raw, val_labels = evaluate(
        model, val_loader, criterion
    )

    # OptimizedRounder tìm ngưỡng tốt nhất trên val set
    print("\n   [OptimizedRounder]:")
    vl_qwk_opt = optR.fit(val_raw, val_labels)

    # scheduler.step() ở đây — không trong train_one_epoch
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    print(f"\nTRAIN : Loss {tr_loss:.4f} | Acc {tr_acc*100:.2f}% | QWK {tr_qwk:.4f}")
    print(f"VAL   : Loss {vl_loss:.4f} | Acc {vl_acc*100:.2f}% | "
          f"QWK(hard) {vl_qwk:.4f} | QWK(opt) {vl_qwk_opt:.4f} | LR {current_lr:.2e}")

    history.append({
        'epoch': epoch+1, 'tr_loss': tr_loss,
        'vl_loss': vl_loss, 'vl_qwk': vl_qwk, 'vl_qwk_opt': vl_qwk_opt
    })

    # Lưu theo QWK tối ưu (không phải hard round)
    if vl_qwk_opt > best_qwk:
        best_qwk     = vl_qwk_opt
        patience_cnt = 0
        torch.save({
            'epoch':       epoch + 1,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'best_qwk':    best_qwk,
            'opt_coef':    optR.coef_.tolist(),   # lưu ngưỡng để dùng lúc inference
            'mode':        'regression'            # flag để cell 6 detect đúng
        }, CFG.save_path)
        print(f"Best model saved. QWK(opt) = {best_qwk:.4f}")
    else:
        patience_cnt += 1
        print(f"No improvement ({patience_cnt}/{CFG.patience})")

    torch.save(model.state_dict(), CFG.last_path)
    torch.cuda.empty_cache()                      # giải phóng VRAM cuối epoch

    if patience_cnt >= CFG.patience:
        print(f"\nEARLY STOPPING. Best QWK(opt) = {best_qwk:.4f}")
        break

# Summary
print(f"\nTRAINING COMPLETED! BEST SCORE: {best_qwk:.4f}")
print(f"Final thresholds: {[f'{c:.4f}' for c in optR.coef_]}")
print(f"\n{'Ep':>3} | {'TR_QWK':>7} | {'VL_QWK':>7} | {'VL_OPT':>7}")
print("-" * 36)
for h in history:
    print(f"{h['epoch']:>3} | {h.get('tr_qwk', 0):>7.4f} | {h['vl_qwk']:>7.4f} | {h['vl_qwk_opt']:>7.4f}")


INIT TRAINING PIPELINE (REGRESSION) - STARTING EPOCH 11...

Epoch [11/30]
      - Batch [50/234] | Loss: 0.2664
      - Batch [100/234] | Loss: 0.1836
      - Batch [150/234] | Loss: 0.1700
      - Batch [200/234] | Loss: 0.3012

   [Confusion Matrix - Validation (hard round)]:
[[3542  269   52    8    1]
 [ 273   66   27    0    0]
 [ 263  157  278   94    2]
 [   9    9   44   64    5]
 [   7   10    9   41   39]]

   [OptimizedRounder]:
  Optimal thresholds: ['0.512', '1.610', '2.147', '3.450']
  Val QWK (optimized): 0.7149

TRAIN : Loss 0.1860 | Acc 67.15% | QWK 0.3085
VAL   : Loss 0.1818 | Acc 75.71% | QWK(hard) 0.7066 | QWK(opt) 0.7149 | LR 2.99e-04
Best model saved. QWK(opt) = 0.7149
Epoch [12/30]
      - Batch [50/234] | Loss: 0.2144
      - Batch [100/234] | Loss: 0.2618
      - Batch [150/234] | Loss: 0.0991
      - Batch [200/234] | Loss: 0.2307

   [Confusion Matrix - Validation (hard round)]:
[[3673  165   29    4    1]
 [ 298   50   18    0    0]
 [ 299  178  222   93   

KeyboardInterrupt: 